# 04 — Segmentation Results: Qualitative Visualisations

Reads pre-computed per-image metrics from `05_test.ipynb` output and produces:

- **Sample prediction figures** (best / worst / random by Dice) — `outputs/figures/.../fold_k/sample_predictions/`
- **Cross-fold summary tables** — `qualitative_fold_summary.csv`, `qualitative_class_summary.csv`, `qualitative_per_image.csv`
- **Dice distribution histogram** — `dice_histogram_by_fold.png`

Rather than re-running full batched inference (which `05_test.ipynb` already did), this notebook reads
`fold_k_test_per_image.csv`, selects the best / worst / random images, and runs inference **only on those
~9 images per fold** to render the overlay figures.

Reads checkpoints and writes outputs **directly to Drive**.  
Reads images from the **local SSD copy** (fast inference).

**Prerequisites:** run `03_train.ipynb` **and** `05_test.ipynb` before running this notebook.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

In [ ]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

## Cell 3 — EXPERIMENT to visualize

Edit `RECIPE`, `DATASET`, and `SPLIT_SCHEME` to match the experiment you trained in `03_train.ipynb`.

Visualization knobs: `FOLDS_TO_VIS`, `N_BEST/WORST/RANDOM_PER_FOLD`, `SHOW_INLINE`.

In [ ]:
import torch
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ── What to visualize ──────────────────────────────────────────────────────
RECIPE       = "03_dicebce"      # one of REFERENCE_RECIPES
DATASET      = "figshare"        # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"    # "image_level"  = §13 reference reproduction (image-level KFold)
                               # "patient_level" = methodologically correct (no patient leakage)

EXPERIMENT = get_experiment(RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1)

assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")

# ── Visualization settings ─────────────────────────────────────────────────
FOLDS_TO_VIS      = [1, 2, 3, 4, 5]  # set [1] first to spot-check
N_BEST_PER_FOLD   = 3
N_WORST_PER_FOLD  = 3
N_RANDOM_PER_FOLD = 3
RANDOM_STATE      = 42
SHOW_INLINE       = True   # False suppresses inline display (faster in long loops)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"experiment   : {EXPERIMENT['name']}")
print(f"dataset      : {EXPERIMENT['dataset']}")
print(f"split_scheme : {EXPERIMENT['split_scheme']}")
print(f"model        : {EXPERIMENT['model_name']}")
print(f"folds        : {FOLDS_TO_VIS}")
print(f"device       : {device}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

## Cell 4 — Copy dataset to local SSD

`copy_to_local` auto-detects a zip at `DRIVE_ROOT/data/<dataset>.zip` and extracts it locally (~2–3 min).  
Falls back to `shutil.copytree` when no zip exists.

Checkpoints and output figures/tables are read from / written to Drive directly (small files, OK over FUSE).

In [ ]:
from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
print(f"LOCAL_ROOT: {LOCAL_ROOT}")

## Cell 5 — `visualize_fold` function

Reads pre-computed per-image metrics from `05_test` output, selects the best / worst / random
images by Dice score, and runs single-image inference on those ~N images to render the overlay figures.

In [ ]:
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

from src.sg_data_utils import build_eval_transform
from src.sg_test_utils import load_model_from_pt, predict_mask
from src.sg_vis_utils  import show_image_gt_pred_overlay
from src.file_utils    import experiment_paths, fold_split_csv_paths, experiment_root_paths
from src.data_utils    import load_metadata


@torch.no_grad()
def visualize_fold(fold: int):
    """
    Reads pre-computed per-image metrics from 05_test output, selects
    best / worst / random images, then runs predict_mask only on those
    ~N images (not the full test set).

    Writes to Drive:
        outputs/tables/<task>/<dataset>/<exp>/fold_<k>/fold_<k>_per_image_dice.csv
        outputs/figures/<task>/<dataset>/<exp>/fold_<k>/sample_predictions/*.png

    Returns the per-image DataFrame (from 05_test output), or None on failure.
    """
    drive_paths = experiment_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
        fold=fold,
    )

    # ── Load pre-computed per-image metrics from 05_test ──────────────────
    per_image_csv = drive_paths["tables"] / f"fold_{fold}_test_per_image.csv"
    if not per_image_csv.exists():
        print(f"  fold {fold}: fold_{fold}_test_per_image.csv not found.")
        print(f"  Run 05_test.ipynb for this experiment first, then re-run this cell.")
        return None

    per_image_df = pd.read_csv(per_image_csv)
    print(f"\n{'='*60}\n  FOLD {fold}  ({len(per_image_df)} test images, metrics from 05_test)\n{'='*60}")
    print(f"  dice  mean={per_image_df['dice'].mean():.4f}  std={per_image_df['dice'].std():.4f}")
    print(f"  iou   mean={per_image_df['iou'].mean():.4f}")

    # ── Select best / worst / random ──────────────────────────────────────
    best  = per_image_df.nlargest (N_BEST_PER_FOLD,   "dice")
    worst = per_image_df.nsmallest(N_WORST_PER_FOLD,  "dice")
    rng   = per_image_df.sample(
        n=min(N_RANDOM_PER_FOLD, len(per_image_df)), random_state=RANDOM_STATE,
    )
    selected = (
        pd.concat([best, worst, rng])
        .drop_duplicates(subset="image_id")
        .reset_index(drop=True)
    )
    print(f"  selected {len(selected)} images for figures "
          f"({N_BEST_PER_FOLD} best / {N_WORST_PER_FOLD} worst / {N_RANDOM_PER_FOLD} random)")

    # ── Load checkpoint (from Drive) ─────────────────────────────────────
    pt_path = drive_paths["best_model"]
    if not pt_path.exists():
        print(f"  best_model.pt not found at {pt_path} — skipping figures")
        return per_image_df

    model, _ = load_model_from_pt(pt_path=pt_path, device=device)
    eval_tf   = build_eval_transform(
        image_size=EXPERIMENT["image_size"],
        preprocessing=EXPERIMENT["preprocessing"],
    )

    fig_dir = drive_paths["figures"] / "sample_predictions"
    fig_dir.mkdir(parents=True, exist_ok=True)

    # ── Run predict_mask only for selected images ─────────────────────────
    t0 = time.time()
    kinds = {"best": best, "worst": worst, "random": rng}
    img_to_kind_rank = {}
    for kind, subset in kinds.items():
        for rank, (_, row) in enumerate(subset.iterrows(), start=1):
            img_to_kind_rank.setdefault(row["image_id"], []).append((kind, rank))

    for _, row in selected.iterrows():
        img_id   = row["image_id"]
        img_path = LOCAL_ROOT / row["image_path"]
        gt_path  = LOCAL_ROOT / row["mask_path"]

        _, pred_mask_arr = predict_mask(
            model, img_path, eval_tf,
            device=device, threshold=EXPERIMENT["threshold"],
        )
        gt_raw = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
        gt_mask_arr = (gt_raw > 127).astype(np.uint8) if gt_raw is not None else np.zeros_like(pred_mask_arr)

        img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img_gray is not None:
            img_gray = cv2.resize(
                img_gray,
                (EXPERIMENT["image_size"], EXPERIMENT["image_size"]),
                interpolation=cv2.INTER_LINEAR,
            )
        else:
            img_gray = np.zeros((EXPERIMENT["image_size"], EXPERIMENT["image_size"]), np.uint8)

        for kind, rank in img_to_kind_rank.get(img_id, [("selected", 1)]):
            show_image_gt_pred_overlay(
                image     = img_gray,
                gt_mask   = gt_mask_arr,
                pred_mask = pred_mask_arr,
                title     = (
                    f"fold {fold} | {kind} #{rank} | id={img_id} "
                    f"| class={row['tumor_class']} | patient={row['patient_id']}"
                ),
                dice      = float(row["dice"]),
                iou       = float(row["iou"]),
                save_path = fig_dir / f"{kind}_{rank:02d}_id{img_id}.png",
                show      = SHOW_INLINE,
            )

    print(f"  figures -> {fig_dir.relative_to(DRIVE_ROOT)}  ({time.time()-t0:.1f}s)")

    # ── Save per-image table (copy of 05_test output, tagged for this nb) ─
    table_csv = drive_paths["tables"] / f"fold_{fold}_per_image_dice.csv"
    per_image_df[["image_id", "patient_id", "tumor_class", "image_path",
                   "mask_path", "dice", "iou"]].to_csv(table_csv, index=False)
    print(f"  table  -> {table_csv.relative_to(DRIVE_ROOT)}")
    return per_image_df


print("visualize_fold defined.")

## Cell 6 — Run visualizations

Iterates over `FOLDS_TO_VIS`. Each fold: loads checkpoint, runs inference, saves figures + table.

In [ ]:
fold_dfs = {}
for fold in FOLDS_TO_VIS:
    fold_dfs[fold] = visualize_fold(fold)

ok_folds = [k for k, v in fold_dfs.items() if v is not None]
print(f"\nDone. {len(ok_folds)}/{len(FOLDS_TO_VIS)} folds visualized: {ok_folds}")

## Cell 7 — Cross-fold summary tables

Aggregates per-fold and per-class Dice / IoU across all visualized folds and writes three CSVs to
`outputs/tables/<task>/<dataset>/<exp>/`.

In [ ]:
from IPython.display import display

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to summarize.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)

    fold_summary  = (
        combined.groupby("fold")[["dice", "iou"]]
        .agg(["mean", "std", "count"])
        .round(4)
    )
    class_summary = (
        combined.groupby("tumor_class")[["dice", "iou"]]
        .agg(["mean", "std", "count"])
        .round(4)
    )

    print("Per-fold Dice / IoU:")
    display(fold_summary)
    print("\nPer-class Dice / IoU (pooled across folds):")
    display(class_summary)

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_dir = root_paths["tables"]
    fold_summary.to_csv (out_dir / "qualitative_fold_summary.csv")
    class_summary.to_csv(out_dir / "qualitative_class_summary.csv")
    combined.to_csv     (out_dir / "qualitative_per_image.csv",   index=False)
    print(f"\nSaved 3 CSVs -> {out_dir.relative_to(DRIVE_ROOT)}")

## Cell 8 — Dice distribution histograms

One histogram per fold, showing the spread of per-image Dice scores.  
Saved as `outputs/figures/<task>/<dataset>/<exp>/dice_histogram_by_fold.png`.

In [ ]:
import matplotlib.pyplot as plt

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to plot.")
else:
    ok_list = [(fold, df) for fold, df in fold_dfs.items() if df is not None]
    n = len(ok_list)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, (fold, df) in zip(axes, ok_list):
        ax.hist(df["dice"].values, bins=20, range=(0, 1),
                color="#2196F3", edgecolor="white", alpha=0.85)
        ax.axvline(df["dice"].mean(), linestyle="--", color="#f44336",
                   label=f"mean={df['dice'].mean():.3f}")
        ax.set_xlabel("Dice score")
        ax.set_title(f"fold {fold}")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

    axes[0].set_ylabel("images (count)")
    fig.suptitle(
        f"{EXPERIMENT['name']} · {EXPERIMENT['dataset']} "
        f"— Dice distribution by fold"
    )
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "dice_histogram_by_fold.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")

## Cell 9 — Per-class Dice bar chart

Mean ± std Dice per tumor class, where error bars reflect cross-fold variance (one mean per fold per class).  
Saved as `outputs/figures/<task>/<dataset>/<exp>/per_class_dice_bar.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to plot.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)

    # One Dice mean per (tumor_class, fold), then summarize across folds.
    # This gives honest error bars that reflect fold-to-fold variance rather
    # than image-level variance (which would understate uncertainty).
    per_fold_class = (
        combined.groupby(["tumor_class", "fold"])["dice"]
        .mean()
        .reset_index()
        .groupby("tumor_class")["dice"]
        .agg(mean="mean", std="std")
        .reset_index()
        .sort_values("tumor_class")
        .reset_index(drop=True)
    )

    classes = per_fold_class["tumor_class"].tolist()
    means   = per_fold_class["mean"].values
    stds    = per_fold_class["std"].fillna(0).values
    n_cls   = len(classes)
    overall_mean = combined["dice"].mean()

    palette = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]
    colors  = palette[:n_cls]

    fig, ax = plt.subplots(figsize=(max(5, n_cls * 2.0), 4.5))
    x    = np.arange(n_cls)
    bars = ax.bar(
        x, means, yerr=stds, capsize=6, color=colors, alpha=0.85, width=0.55,
        error_kw={"elinewidth": 1.5, "capthick": 1.5},
    )

    for bar, m, s in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            m + s + 0.012,
            f"{m:.3f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )

    ax.axhline(overall_mean, linestyle="--", color="#777",
               label=f"overall mean = {overall_mean:.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=11)
    ax.set_ylabel("Dice  (mean ± std, cross-fold)")
    ax.set_ylim(0, 1.08)
    ax.set_title(
        f"{EXPERIMENT['name']}  ·  {EXPERIMENT['dataset']} "
        f"— Dice by tumor class  (n={len(ok_folds)} folds)"
    )
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "per_class_dice_bar.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")